# 01 — Data Exploration

This notebook loads the preprocessed TruthfulQA and HaluEval datasets,
inspects class balance, token length distributions, and displays sample data.

**Prerequisites**: Run `python data/prepare_datasets.py` from the project root first
(or this notebook will run it for you).

In [ ]:
import sys
import os
from pathlib import Path

# Ensure project root is on the path
project_root = Path(os.getcwd()).parent if 'notebooks' in os.getcwd() else Path(os.getcwd())
sys.path.insert(0, str(project_root))
os.chdir(project_root)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Style setup
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (12, 5)

print(f'Project root: {project_root}')
print(f'Python: {sys.version}')

## 1. Load Datasets

If the CSVs don't exist yet, we'll run `data/prepare_datasets.py` automatically.

In [ ]:
# Check if data exists, if not run the preparation script
data_dir = project_root / 'data'
tqa_train_path = data_dir / 'truthfulqa' / 'train.csv'
halu_train_path = data_dir / 'halueval' / 'train.csv'

if not tqa_train_path.exists() or not halu_train_path.exists():
    print('⚠️  CSVs not found — running data preparation script...')
    !python data/prepare_datasets.py
    print('\n✅ Data preparation complete!')
else:
    print('✅ Data files found!')

# Load all splits
datasets_info = {}
for ds_name in ['truthfulqa', 'halueval']:
    datasets_info[ds_name] = {}
    for split in ['train', 'val', 'test']:
        path = data_dir / ds_name / f'{split}.csv'
        df = pd.read_csv(path)
        datasets_info[ds_name][split] = df
        print(f'  {ds_name}/{split}: {len(df):>6,} samples')

print(f'\n📊 Total samples loaded: {sum(len(df) for ds in datasets_info.values() for df in ds.values()):,}')

## 2. Dataset Size & Split Distribution

In [ ]:
# Summary table
rows = []
for ds_name, splits in datasets_info.items():
    for split_name, df in splits.items():
        n0 = (df.label == 0).sum()
        n1 = (df.label == 1).sum()
        rows.append({
            'Dataset': ds_name.upper(),
            'Split': split_name,
            'Total': len(df),
            'Truthful (0)': n0,
            'Hallucinated (1)': n1,
            'Balance': f'{n0/len(df):.1%} / {n1/len(df):.1%}',
        })

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))

## 3. Class Balance Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, (ds_name, splits) in enumerate(datasets_info.items()):
    ax = axes[idx]
    split_names = list(splits.keys())
    truthful_counts = [(s.label == 0).sum() for s in splits.values()]
    hallucinated_counts = [(s.label == 1).sum() for s in splits.values()]
    
    x = np.arange(len(split_names))
    width = 0.35
    
    bars1 = ax.bar(x - width/2, truthful_counts, width, label='Truthful (0)', color='#2ecc71', alpha=0.85)
    bars2 = ax.bar(x + width/2, hallucinated_counts, width, label='Hallucinated (1)', color='#e74c3c', alpha=0.85)
    
    ax.set_xlabel('Split')
    ax.set_ylabel('Count')
    ax.set_title(f'{ds_name.upper()} — Class Balance per Split', fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels([s.capitalize() for s in split_names])
    ax.legend()
    
    # Add count labels on bars
    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 5,
                f'{int(bar.get_height()):,}', ha='center', va='bottom', fontsize=9)
    for bar in bars2:
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 5,
                f'{int(bar.get_height()):,}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('results/plots/class_balance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/plots/class_balance.png')

## 4. Text Length Distribution

We measure text length in characters and approximate token count (words).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for row, (ds_name, splits) in enumerate(datasets_info.items()):
    # Combine all splits for distribution analysis
    all_data = pd.concat(splits.values(), ignore_index=True)
    all_data['char_length'] = all_data['text'].str.len()
    all_data['word_count'] = all_data['text'].str.split().str.len()
    
    # Character length histogram
    ax = axes[row, 0]
    for label, color, name in [(0, '#2ecc71', 'Truthful'), (1, '#e74c3c', 'Hallucinated')]:
        subset = all_data[all_data.label == label]
        ax.hist(subset['char_length'], bins=50, alpha=0.6, color=color, label=name, edgecolor='white')
    ax.set_xlabel('Character Length')
    ax.set_ylabel('Frequency')
    ax.set_title(f'{ds_name.upper()} — Character Length Distribution', fontweight='bold')
    ax.legend()
    
    # Word count histogram
    ax = axes[row, 1]
    for label, color, name in [(0, '#2ecc71', 'Truthful'), (1, '#e74c3c', 'Hallucinated')]:
        subset = all_data[all_data.label == label]
        ax.hist(subset['word_count'], bins=50, alpha=0.6, color=color, label=name, edgecolor='white')
    ax.set_xlabel('Word Count (approx. tokens)')
    ax.set_ylabel('Frequency')
    ax.set_title(f'{ds_name.upper()} — Word Count Distribution', fontweight='bold')
    ax.legend()
    
    # Print stats
    print(f'\n{ds_name.upper()} text stats:')
    print(f'  Char length — mean: {all_data["char_length"].mean():.0f}, '
          f'median: {all_data["char_length"].median():.0f}, '
          f'max: {all_data["char_length"].max()}')
    print(f'  Word count  — mean: {all_data["word_count"].mean():.0f}, '
          f'median: {all_data["word_count"].median():.0f}, '
          f'max: {all_data["word_count"].max()}')

plt.tight_layout()
plt.savefig('results/plots/text_length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nSaved → results/plots/text_length_distribution.png')

## 5. Sample Inspection

Let's look at some random truthful and hallucinated examples from each dataset.

In [ ]:
np.random.seed(42)

for ds_name, splits in datasets_info.items():
    train_df = splits['train']
    print(f'\n{"="*70}')
    print(f'  {ds_name.upper()} — Sample Inspection (from train split)')
    print(f'{"="*70}')
    
    for label, emoji, desc in [(0, '✅', 'TRUTHFUL'), (1, '❌', 'HALLUCINATED')]:
        subset = train_df[train_df.label == label]
        samples = subset.sample(n=min(3, len(subset)), random_state=42)
        print(f'\n  {emoji} {desc} examples:')
        print(f'  {"-"*50}')
        for i, (_, row) in enumerate(samples.iterrows(), 1):
            # Show the text, truncating if too long
            text = row['text']
            if len(text) > 200:
                text = text[:200] + '...'
            print(f'  [{i}] {text}')
            print()

## 6. Summary Statistics Table

In [ ]:
# Create a comprehensive summary table
stats_rows = []

for ds_name, splits in datasets_info.items():
    all_data = pd.concat(splits.values(), ignore_index=True)
    all_data['char_length'] = all_data['text'].str.len()
    all_data['word_count'] = all_data['text'].str.split().str.len()
    
    for label, desc in [(0, 'Truthful'), (1, 'Hallucinated')]:
        subset = all_data[all_data.label == label]
        stats_rows.append({
            'Dataset': ds_name.upper(),
            'Class': desc,
            'Count': len(subset),
            'Avg Chars': f'{subset["char_length"].mean():.0f}',
            'Median Chars': f'{subset["char_length"].median():.0f}',
            'Max Chars': subset['char_length'].max(),
            'Avg Words': f'{subset["word_count"].mean():.0f}',
            'Median Words': f'{subset["word_count"].median():.0f}',
            'Max Words': subset['word_count'].max(),
        })

stats_df = pd.DataFrame(stats_rows)
print(stats_df.to_string(index=False))

## ✅ Phase 1 Complete

Datasets are preprocessed and saved. **Next steps:**

1. **Phase 2** — Upload processed CSVs to Google Colab, load Llama-3.1-8B, extract hidden states
2. **Phase 3** — Download `hidden_states.h5` back to local, train probe classifiers (CPU-only)

See `notebooks/02_hidden_state_extraction_colab.ipynb` for the Colab notebook.